In [2]:
print("--- Cell 1: Fixing Environment ---")
# 1. Uninstall conflicting libraries that also use pyarrow
print("Uninstalling conflicting libraries (cudf, bigframes, etc.)...")
!pip uninstall -y cudf pylibcudf-cu12 cudf-cu12 bigframes

--- Cell 1: Fixing Environment ---
Uninstalling conflicting libraries (cudf, bigframes, etc.)...
Found existing installation: pylibcudf-cu12 25.2.2
Uninstalling pylibcudf-cu12-25.2.2:
  Successfully uninstalled pylibcudf-cu12-25.2.2
Found existing installation: cudf-cu12 25.2.2
Uninstalling cudf-cu12-25.2.2:
  Successfully uninstalled cudf-cu12-25.2.2
Found existing installation: bigframes 2.12.0
Uninstalling bigframes-2.12.0:
  Successfully uninstalled bigframes-2.12.0


In [3]:
print("Installing our project libraries...")
!pip install -q datasets transformers pandas tqdm ipywidgets
print("Done.\n")

Installing our project libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 11.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 38.7 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.0a1 which is incompatible.
pandas-gbq 0.29.2 requires google-api-core<3.0.0,>=2.10.2, but you have google-api-core 1.34.1 which is incompatible.
Done.



In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from datasets import load_dataset
import pandas as pd
from tqdm.notebook import tqdm
import numpy as np
import os
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

2025-11-05 14:52:57.458118: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762354377.635744      46 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762354377.690056      46 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [5]:
def get_device():
    """
    Helper function to get the correct device for training.
    Will detect Kaggle's CUDA GPU automatically.
    """
    if torch.backends.mps.is_available():
        print("Apple MPS (M-series GPU) is available. Using MPS.")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("NVIDIA CUDA GPU is available. Using CUDA.")
        return torch.device("cuda")
    else:
        print("No GPU detected. Using CPU.")
        return torch.device("cpu")

In [6]:
CONFIG = {
    "snli_samples": 550_152,       # Use all of SNLI
    "mnli_samples": 392_702,       # Use all of MNLI
    "model_name": "bert-base-uncased",
    "max_length": 64,       # Shorter sequence length saves memory
    "batch_size": 64,       # T4 GPUs can handle this
    "accumulation_steps": 2, # Effective batch size = 128
    "epochs": 3,            # 3 epochs for the full dataset
    "learning_rate": 2e-5,
    "warmup_ratio": 0.1,    # 10% of steps for warmup
    "device": get_device()
}


NVIDIA CUDA GPU is available. Using CUDA.


In [7]:
print(f"Effective batch size: {CONFIG['batch_size'] * CONFIG['accumulation_steps']}")
print(f"Training for {CONFIG['epochs']} epochs.")
print("Done.\n")

Effective batch size: 128
Training for 3 epochs.
Done.



In [8]:
def create_triplets_from_dataset(dataset_name, n_samples, config_name=None, split="train"):
    """
    Loads a dataset, converts to pandas, and groups to find (A, P, N) triplets.
    """
    print(f"Loading {dataset_name}" + (f" (config: {config_name})" if config_name else "") + "...")
    
    # Load a slice of the dataset
    dataset = load_dataset(dataset_name, config_name, split=f'{split}[:{n_samples}]')
    df = dataset.to_pandas()
    
    # Filter out neutral (label 1) and any NaNs
    df = df[df['label'] != 1]
    df = df.dropna(subset=['premise', 'hypothesis'])
    
    print("Grouping by premise to find (Anchor, Positive, Negative) pairs...")
    grouped = df.groupby('premise')
    triplets = []
    
    desc = f"Processing {dataset_name}" + (f" ({config_name})" if config_name else "")
    # Use notebook-friendly tqdm
    for premise, group in tqdm(grouped, desc=desc):
        positives = group[group['label'] == 0]['hypothesis'].tolist()
        negatives = group[group['label'] == 2]['hypothesis'].tolist()
        
        # Only add if we have at least one of each
        if positives and negatives:
            triplets.append((premise, positives[0], negatives[0]))
            
    print(f"Found {len(triplets)} triplets in {dataset_name}" + (f" ({config_name})" if config_name else "") + ".")
    return triplets

In [9]:
snli_triplets = create_triplets_from_dataset(
    'snli', 
    n_samples=CONFIG['snli_samples']
)

# Load and process MNLI
mnli_triplets = create_triplets_from_dataset(
    'glue', 
    n_samples=CONFIG['mnli_samples'], 
    config_name='mnli'
)

Loading snli...


README.md: 0.00B [00:00, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/412k [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/413k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

...

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

Grouping by premise to find (Anchor, Positive, Negative) pairs...


Processing glue (mnli):   0%|          | 0/128142 [00:00<?, ?it/s]

Found 128132 triplets in glue (mnli).


In [10]:
all_triplets = snli_triplets + mnli_triplets


In [11]:
class TripletDataset(Dataset):
    def __init__(self, triplets, tokenizer, max_length=CONFIG['max_length']):
        self.triplets = triplets
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        anchor, positive, negative = self.triplets[idx]
        return {
            "anchor": anchor,
            "positive": positive,
            "negative": negative
        }

In [12]:
def collate_fn(batch):
    anchors = [item['anchor'] for item in batch]
    positives = [item['positive'] for item in batch]
    negatives = [item['negative'] for item in batch]
    
    tok_anchor = tokenizer(anchors, padding=True, truncation=True, 
                           max_length=CONFIG['max_length'], return_tensors='pt')
    tok_positive = tokenizer(positives, padding=True, truncation=True, 
                             max_length=CONFIG['max_length'], return_tensors='pt')
    tok_negative = tokenizer(negatives, padding=True, truncation=True, 
                             max_length=CONFIG['max_length'], return_tensors='pt')
    
    return {
        "anchor": tok_anchor,
        "positive": tok_positive,
        "negative": tok_negative
    }

In [13]:
tokenizer = BertTokenizer.from_pretrained(CONFIG['model_name'])
train_dataset = TripletDataset(all_triplets, tokenizer)
train_dataloader = DataLoader(train_dataset, 
                            batch_size=CONFIG['batch_size'], 
                            shuffle=True, 
                            collate_fn=collate_fn)
print(f"DataLoader created with {len(train_dataloader)} batches.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

DataLoader created with 4333 batches.


In [14]:
class SimCSEModel(nn.Module):
    def __init__(self, model_name=CONFIG['model_name']):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
    
    # Helper function for mean pooling
    def mean_pooling(self, model_output, attention_mask):
        # model_output[0] is the last_hidden_state
        token_embeddings = model_output[0] 
        
        # Expand attention mask to match token embeddings dimensions
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        
        # Sum embeddings, masked by attention mask
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        
        # Get count of non-padding tokens (clamp to avoid division by zero)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9) 
        
        # Return averaged embedding
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask, token_type_ids):
        # Get BERT outputs
        outputs = self.bert(input_ids=input_ids, 
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
        
        # --- Apply Mean Pooling ---
        pooled_output = self.mean_pooling(outputs, attention_mask)
        
        # Normalize the embedding
        return F.normalize(pooled_output, p=2, dim=1)

model = SimCSEModel().to(CONFIG['device'])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [15]:
def contrastive_loss(emb_a, emb_p, emb_n, temperature=0.05):
    # emb_a, emb_p, emb_n are all shape (batch_size, embed_dim)
    batch_size = emb_a.shape[0]
    
    # 1. Calculate similarities
    
    # Calculate sim(A, P)
    # We want a (batch_size, batch_size) matrix where the diagonal
    # holds the A[i] vs P[i] similarity.
    sim_ap = F.cosine_similarity(emb_a.unsqueeze(1), emb_p.unsqueeze(0), dim=-1)
    
    # Calculate sim(A, N)
    # This will also be a (batch_size, batch_size) matrix
    sim_an = F.cosine_similarity(emb_a.unsqueeze(1), emb_n.unsqueeze(0), dim=-1)
    
    # 2. Concatenate P and N to form all candidates
    # The logits for each A[i] will be:
    # [sim(A_i, P_0), ..., sim(A_i, P_N), sim(A_i, N_0), ..., sim(A_i, N_N)]
    # Shape: (batch_size, 2 * batch_size)
    logits = torch.cat([sim_ap, sim_an], dim=1)
    
    # Apply temperature scaling
    logits /= temperature
    
    # 3. Create labels
    # The positive example for anchor `i` is positive `i`.
    # In our `logits` matrix, this is at index `i`.
    labels = torch.arange(batch_size, device=emb_a.device)
    
    # 4. Calculate CrossEntropy
    # The model must pick the correct P[i] (at index i) from the
    # 2*B candidates (B positives and B negatives).
    return nn.CrossEntropyLoss()(logits, labels)

loss_fn = contrastive_loss

In [16]:
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'])

# Calculate total training steps for the scheduler
total_steps = (len(train_dataloader) // CONFIG['accumulation_steps']) * CONFIG['epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])

In [17]:
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [18]:
model.train()
optimizer.zero_grad() # Zero gradients at the start

for epoch in range(CONFIG['epochs']):
    print(f"\n--- Epoch {epoch + 1}/{CONFIG['epochs']} ---")
    
    total_loss = 0
    progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc="Training")
    
    for i, batch in progress_bar:
        # 1. Move all parts of the batch to the device
        anc_inputs = {k: v.to(CONFIG['device']) for k, v in batch['anchor'].items()}
        pos_inputs = {k: v.to(CONFIG['device']) for k, v in batch['positive'].items()}
        neg_inputs = {k: v.to(CONFIG['device']) for k, v in batch['negative'].items()}
        
        # 2. Get embeddings for all three
        emb_a = model(**anc_inputs)
        emb_p = model(**pos_inputs)
        emb_n = model(**neg_inputs)
        
        # 3. Compute loss (using the new V3 loss function)
        loss = loss_fn(emb_a, emb_p, emb_n)
        
        # 4. Gradient Accumulation: Normalize loss and backpropagate
        loss = loss / CONFIG['accumulation_steps']
        loss.backward()
        
        # 5. Gradient Accumulation: Step and clear
        if (i + 1) % CONFIG['accumulation_steps'] == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Added gradient clipping
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
        total_loss += loss.item() * CONFIG['accumulation_steps'] # De-normalize loss for logging
        
        # Update progress bar
        progress_bar.set_postfix({'loss': total_loss / (i + 1)})

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")



--- Epoch 1/3 ---


Training:   0%|          | 0/4333 [00:00<?, ?it/s]

Epoch 1 Average Loss: 0.4853

--- Epoch 2/3 ---


Training:   0%|          | 0/4333 [00:00<?, ?it/s]

Epoch 2 Average Loss: 0.2512

--- Epoch 3/3 ---


Training:   0%|          | 0/4333 [00:00<?, ?it/s]

Epoch 3 Average Loss: 0.1949


In [19]:
output_dir = "./my_v3_simcse_model"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"Saving model to {output_dir}...")
model.bert.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

Saving model to ./my_v3_simcse_model...


('./my_v3_simcse_model/tokenizer_config.json',
 './my_v3_simcse_model/special_tokens_map.json',
 './my_v3_simcse_model/vocab.txt',
 './my_v3_simcse_model/added_tokens.json')

In [20]:
sts = load_dataset("stsb_multi_mt", name="en", split="test")
sts = pd.DataFrame(sts)
print(f"Loaded {len(sts)} samples from STSb test set.")

README.md: 0.00B [00:00, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/470k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/108k [00:00<?, ?B/s]

en/dev-00000-of-00001.parquet:   0%|          | 0.00/142k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Loaded 1379 samples from STSb test set.


In [21]:
model.eval() 

SimCSEModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [22]:
def embed_texts(texts):
    if isinstance(texts, pd.Series):
        texts = texts.tolist()
        
    enc = tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        return_tensors='pt', 
        max_length=CONFIG['max_length']
    ).to(CONFIG['device'])
    
    with torch.no_grad():
        # ** Use the model with its correct V3 forward pass **
        return model(
            input_ids=enc['input_ids'], 
            attention_mask=enc['attention_mask'],
            token_type_ids=enc['token_type_ids']
        ).cpu()

In [23]:
emb1 = embed_texts(sts['sentence1'])

emb2 = embed_texts(sts['sentence2'])



In [24]:
similarities = cosine_similarity(emb1, emb2).diagonal()

In [25]:
corr = spearmanr(similarities, sts['similarity_score'])

print(f"Spearman Correlation: {corr.correlation:.4f}")

Spearman Correlation: 0.8524
